# SoulTuner：AMD ROCm 上的 Evidence-first 音乐检索 Planner

**ModelScope Gallery / 灵感流代码实践**  
建议标签：`AMD GPU激励计划`、`ROCm`、`LLM 微调`、`音乐推荐`、`Graph RAG`

本 Notebook 使用公开合成请求演示 SoulTuner Planner 的核心设计：先抽取可审计证据，再决定 Graph / Dense / Web 的角色，最后由确定性编译器生成执行权重。默认 smoke mode 不需要私有数据、模型权重或 API Key，所有 Cell 均可运行；如果配置了真实 Planner HTTPS 端点，最后一节会额外测试 35B 候选模型。完整工程见 [SoulTuner-Agent](https://github.com/hgsanyang/SoulTuner-Agent)。

在完整项目中，本 Notebook 只聚焦可独立部署的 Planner 切片：`用户请求 → Planner → Graph / Dense / Web → 结果融合 → 长期记忆与反馈`。Graph 对接实体、目录与标签，Dense 对接音乐音频向量；推荐融合、Neo4j/Qdrant、用户反馈和记忆仍由 SoulTuner-Agent 主工程承接。

> `brief_reason` 是不超过 80 字的公开说明，不是隐藏思维链；执行只读取结构化字段。

## 1. 检测 AMD ROCm 运行环境

ModelScope AMD 创空间或 MI308X 实例通常会暴露 `/dev/kfd`。代码同时尝试读取 `rocminfo` 和 PyTorch HIP 信息；在 CPU 环境中会安全跳过，不会报错。

In [1]:
import json
import os
import platform
import re
import shutil
import subprocess
import time
import urllib.error
import urllib.request
from copy import deepcopy

runtime = {
    'python': platform.python_version(),
    'dev_kfd': os.path.exists('/dev/kfd'),
    'rocminfo': shutil.which('rocminfo'),
}
if runtime['rocminfo']:
    probe = subprocess.run([runtime['rocminfo']], capture_output=True, text=True, timeout=5, check=False)
    runtime['amd_devices'] = list(dict.fromkeys(
        name for line in probe.stdout.splitlines()
        if 'Marketing Name:' in line
        and (name := line.split(':', 1)[1].strip())
        and 'intel' not in name.casefold()
    ))
try:
    import torch
    runtime.update({
        'torch': torch.__version__,
        'hip': torch.version.hip,
        'gpu_available': torch.cuda.is_available(),
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    })
except Exception as exc:
    runtime['torch_probe'] = f'skipped: {type(exc).__name__}'
print(json.dumps(runtime, ensure_ascii=False, indent=2))

{
  "python": "3.13.5",
  "dev_kfd": false,
  "rocminfo": null,
  "torch_probe": "skipped: ModuleNotFoundError"
}


## 2. Graph 和 Dense 不是二选一

- **Graph**：歌手、歌曲、语言、年代、流派、情绪/场景标签等目录事实。
- **Dense**：参考歌曲相似度、音色、鼓点、低音、空间感和主观听感。
- **Web**：最新发行、热榜、新闻等时效信息。

`required` 表示没有该通道就无法满足请求；`optional` 只提供辅助证据；`off` 表示不应调用。以下实现是可独立运行的教学版安全 Planner。

In [2]:
MOODS = {'开心', '快乐', '治愈', '温暖', '难过', '伤心', '低落', '焦虑', '平静', '浪漫', '孤独'}
GENRES = {'摇滚', '爵士', '古典', '民谣', '电子', '说唱', '嘻哈', 'R&B', '金属', '流行'}
SCENARIOS = {'跑步', '健身', '学习', '工作', '睡前', '通勤', '开车', '聚会', '旅行'}
ACOUSTIC = {'bass', 'base', '低音', '贝斯', '鼓声', '鼓点', '人声', '吉他', '钢琴', '混响', '空间感', '动态', '节奏', '音色', '听感', '氛围'}
REFERENCES = {'刚刚那首', '刚才那首', '上一首', '这首歌', '类似这首', '相似的歌'}
FRESH = {'最新', '最近', '刚发行', '本周', '今年', '热榜', '实时'}
GUIDANCE = {'怎么使用', '如何使用', '怎么导入', '如何导入', '收藏在哪', '歌单在哪', '怎么设置'}

def contains(text, terms):
    lowered = text.casefold()
    return any(term.casefold() in lowered for term in terms)

def matched(text, terms):
    lowered = text.casefold()
    return [term for term in sorted(terms) if term.casefold() in lowered]

def compile_execution(policy):
    profiles = {
        ('required', 'off'): ('graph_only', 1.0, 0.0),
        ('required', 'optional'): ('graph_primary', 0.75, 0.25),
        ('required', 'required'): ('balanced_hybrid', 0.5, 0.5),
        ('optional', 'required'): ('dense_primary', 0.25, 0.75),
        ('off', 'required'): ('dense_only', 0.0, 1.0),
        ('off', 'off'): ('no_retrieval', 0.0, 0.0),
    }
    profile, graph_weight, dense_weight = profiles[(policy['graph'], policy['dense'])]
    return {
        'profile': profile,
        'tools': [lane for lane in ('graph', 'dense', 'web') if policy[lane] != 'off'],
        'weights': {'graph': graph_weight, 'dense': dense_weight},
    }

def safe_plan(text, reference_title='', reference_artist=''):
    text = str(text or '').strip()
    policy = {'graph': 'off', 'dense': 'off', 'web': 'off'}
    result = {
        'task_mode': 'recommendation', 'dialogue_mode': None, 'response_mode': 'answer',
        'evidence': {'reason_codes': [], 'reference_songs': [], 'brief_reason': ''},
        'lane_policy': policy, 'hints': {'mood': [], 'scenario': [], 'genre': []},
        'acoustic_queries': [], 'clarification': None,
    }
    if not text:
        result.update(task_mode='dialogue', dialogue_mode='chat', response_mode='clarify')
        result['clarification'] = '请告诉我你想听什么样的音乐。'
        result['evidence'].update(reason_codes=['underspecified_request'], brief_reason='请求为空，需要补充音乐需求')
    elif contains(text, GUIDANCE):
        result.update(task_mode='dialogue', dialogue_mode='library_guidance')
        result['evidence'].update(reason_codes=['no_retrieval_needed'], brief_reason='产品使用问题不应触发音乐召回')
    elif contains(text, REFERENCES) and not reference_title:
        result['response_mode'] = 'clarify'
        result['clarification'] = '请提供你指的参考歌曲。'
        result['evidence'].update(reason_codes=['unresolved_reference'], brief_reason='参考歌曲无法唯一解析')
    else:
        moods, genres, scenarios = matched(text, MOODS), matched(text, GENRES), matched(text, SCENARIOS)
        has_acoustic = contains(text, ACOUSTIC)
        has_affective = bool(moods) or contains(text, {'感觉', '让我', '陪我', '适合我'})
        dense = has_acoustic or has_affective or bool(reference_title)
        graph = bool(moods or genres or scenarios)
        result['hints'] = {'mood': moods, 'scenario': scenarios, 'genre': genres}
        codes = []
        if moods:
            codes.append('taggable_mood')
        if genres:
            codes.append('taggable_genre')
        if scenarios:
            codes.append('taggable_scenario')
        if has_affective:
            codes.append('subjective_affective_goal')
        if has_acoustic:
            codes.append('acoustic_timbre_or_instrument')
        if reference_title:
            codes += ['reference_track_similarity', 'resolved_context_reference']
            result['evidence']['reference_songs'] = [{'title': reference_title, 'artist': reference_artist or None, 'source': 'previous_results'}]
        if dense:
            policy['dense'] = 'required'
            if not reference_title:
                result['acoustic_queries'] = [text]
        if graph:
            policy['graph'] = 'optional' if dense else 'required'
        if contains(text, FRESH):
            policy['web'] = 'required'
            if policy['graph'] == 'off':
                policy['graph'] = 'optional'
            codes.append('freshness_or_external')
        if all(v == 'off' for v in policy.values()):
            policy['dense'], result['acoustic_queries'] = 'required', [text]
            codes.append('metaphorical_vibe')
        result['evidence']['reason_codes'] = list(dict.fromkeys(codes))
        if policy['graph'] == 'optional' and policy['dense'] == 'required':
            reason = '标签可辅助粗筛，主观听感与相似度由声学检索主导'
        elif policy['graph'] == 'required':
            reason = '目录标签可直接约束候选，优先使用图谱检索'
        else:
            reason = '请求描述的是听感或声学特征，应以向量召回为主'
        result['evidence']['brief_reason'] = reason[:80]
    result['execution'] = compile_execution(policy)
    return result

print('教学版 Planner 已加载')

教学版 Planner 已加载


## 3. 运行五类代表性请求

重点观察“心情很差”与“低音更重”的差别：前者允许 Graph 用情绪标签辅助，后者才接近真正的 Dense-only。

In [3]:
examples = [
    ('情绪 + 听感', '我今天心情很差，想听温暖治愈的歌', '', ''),
    ('纯声学', '我希望 bass 更重、鼓声更大一些', '', ''),
    ('图谱标签', '找一些适合学习的爵士', '', ''),
    ('参考歌曲', '给我和刚刚那首歌听感相似的', 'Dreams', 'Fleetwood Mac'),
    ('产品指导', '怎么导入我的网易云歌单？', '', ''),
]
for name, text, title, artist in examples:
    plan = safe_plan(text, title, artist)
    print(f"\n[{name}] {text}")
    print('policy =', plan['lane_policy'], '| profile =', plan['execution']['profile'])
    print('reason =', plan['evidence']['brief_reason'])
    if plan['evidence']['reference_songs']:
        print('reference =', plan['evidence']['reference_songs'])


[情绪 + 听感] 我今天心情很差，想听温暖治愈的歌
policy = {'graph': 'optional', 'dense': 'required', 'web': 'off'} | profile = dense_primary
reason = 标签可辅助粗筛，主观听感与相似度由声学检索主导

[纯声学] 我希望 bass 更重、鼓声更大一些
policy = {'graph': 'off', 'dense': 'required', 'web': 'off'} | profile = dense_only
reason = 请求描述的是听感或声学特征，应以向量召回为主

[图谱标签] 找一些适合学习的爵士
policy = {'graph': 'required', 'dense': 'off', 'web': 'off'} | profile = graph_only
reason = 目录标签可直接约束候选，优先使用图谱检索

[参考歌曲] 给我和刚刚那首歌听感相似的
policy = {'graph': 'off', 'dense': 'required', 'web': 'off'} | profile = dense_only
reason = 请求描述的是听感或声学特征，应以向量召回为主
reference = [{'title': 'Dreams', 'artist': 'Fleetwood Mac', 'source': 'previous_results'}]

[产品指导] 怎么导入我的网易云歌单？
policy = {'graph': 'off', 'dense': 'off', 'web': 'off'} | profile = no_retrieval
reason = 产品使用问题不应触发音乐召回


## 4. Fail-closed 候选守卫

真实部署中，35B 模型只提出候选。如果它把“低音更重”错误地路由成 Graph-only，守卫必须在执行前拒绝。

In [4]:
def guard_candidate(text, candidate, reference_title='', reference_artist=''):
    minimum = safe_plan(text, reference_title, reference_artist)
    findings = []
    if not isinstance(candidate, dict) or not isinstance(candidate.get('lane_policy'), dict):
        findings.append('候选结构无效')
    else:
        for lane, role in minimum['lane_policy'].items():
            if role == 'required' and candidate['lane_policy'].get(lane) != 'required':
                findings.append(f'遗漏必需的 {lane} 通道')
        if candidate.get('task_mode') != minimum['task_mode']:
            findings.append('任务类型冲突')
    if findings:
        minimum['source'] = 'deterministic_fallback'
        return minimum, findings + ['候选被拒绝']
    accepted = deepcopy(candidate)
    accepted['source'] = 'model_candidate_guarded'
    accepted['execution'] = compile_execution(accepted['lane_policy'])
    return accepted, ['候选通过守卫']

request = '我希望 bass 更重、鼓声更大一些'
bad_candidate = safe_plan(request)
bad_candidate['lane_policy'] = {'graph': 'required', 'dense': 'off', 'web': 'off'}
guarded, findings = guard_candidate(request, bad_candidate)
print('findings:', findings)
print('executed policy:', guarded['lane_policy'])
assert guarded['lane_policy']['dense'] == 'required'
assert guarded['source'] == 'deterministic_fallback'

findings: ['遗漏必需的 dense 通道', '候选被拒绝']
executed policy: {'graph': 'off', 'dense': 'required', 'web': 'off'}


## 5. 微调 35B 与 Qwen3.7 Plus API 基线

35B-A3B Planner 在 AMD MI308X 上完成 2 epoch 训练。Qwen3.7 Plus API 与微调模型使用同一份 sealed 500、同一 V5 prompt 和同一确定性 scorer，原始输出不做修复。下面只展示聚合准确率，不包含私有 sealed 样本。Lane policy 检查必需/禁止通道是否满足；Lane role exact 要求 Graph、Dense、Web 的 required / optional / off 逐项完全一致。该对比仅说明 SoulTuner Planner 任务适配度，不代表通用能力排名。

In [5]:
metrics = {
    'Schema valid': (99.51, 99.40),
    'Compile success': (99.51, 99.20),
    'Intent / route': (99.03, 95.60),
    'Lane policy': (91.75, 86.80),
    'Lane role': (78.88, 76.20),
    'HyDE': (98.88, 84.44),
}
print(f"{'Metric':<20} {'Regression 412':>16} {'Sealed 500':>12} {'Gap':>9}")
print('-' * 62)
for metric, (regression, sealed) in metrics.items():
    print(f"{metric:<20} {regression:>15.2f}% {sealed:>11.2f}% {sealed-regression:>8.2f}pp")
print('\nSealed required-lane recall: 89.98%')
print('Thinking non-empty: 0')
qwen_baseline = {
    'Schema valid': 57.60,
    'Compile success': 57.40,
    'Intent / route': 52.80,
    'Lane policy': 48.40,
    'Lane role': 42.40,
    'HyDE': 48.89,
}
print(f"\n{'Sealed 500':<20} {'Fine-tuned 35B':>16} {'Qwen3.7 Plus':>16} {'Gain':>10}")
print('-' * 66)
for metric, qwen_value in qwen_baseline.items():
    tuned_value = metrics[metric][1]
    print(f"{metric:<20} {tuned_value:>15.2f}% {qwen_value:>15.2f}% {tuned_value-qwen_value:>9.2f}pp")
print(f"{'Required lane':<20} {89.98:>15.2f}% {32.33:>15.2f}% {57.65:>9.2f}pp")

Metric                 Regression 412   Sealed 500       Gap
--------------------------------------------------------------
Schema valid                   99.51%       99.40%    -0.11pp
Compile success                99.51%       99.20%    -0.31pp
Intent / route                 99.03%       95.60%    -3.43pp
Lane policy                    91.75%       86.80%    -4.95pp
Lane role                      78.88%       76.20%    -2.68pp
HyDE                           98.88%       84.44%   -14.44pp

Sealed required-lane recall: 89.98%
Thinking non-empty: 0

Sealed 500             Fine-tuned 35B     Qwen3.7 Plus       Gain
------------------------------------------------------------------
Schema valid                   99.40%           57.60%     41.80pp
Compile success                99.20%           57.40%     41.80pp
Intent / route                 95.60%           52.80%     42.80pp
Lane policy                    86.80%           48.40%     38.40pp
Lane role                      76.20%      

## 6. AMD 真实部署证据与可选端点调用

2026-08-12 在 AMD MI308X 实例上以 `checkpoint-450` 启动 OpenAI-compatible endpoint，并通过同一 Gradio/Guard 路径执行公开请求。热启动端到端耗时 24.56 秒，结果来源为 `model_candidate_guarded`，Graph/Dense 权重为 0.25/0.75，非空 thinking 为 0。这个记录只包含公开请求与聚合运行信息。

创空间应用提供 Qwen3.7 Plus、SoulTuner V4.2 35B 和安全演示三个档位。本 Cell 使用 `SOULTUNER_PLANNER_ENDPOINT` 与可选的 `SOULTUNER_PLANNER_TOKEN`；未配置时明确跳过，Notebook 仍算成功执行。`SOULTUNER_PLANNER_PROTOCOL=openai` 对接 `/v1/chat/completions`，设为 `planner` 可对接直接返回 V5 JSON 或 `{"decision": {...}}` 的业务端点。

In [6]:
verified_deployment = {
    'run_id': 'planner-v4.2-35b-2ep-20260810T164032Z',
    'gpu': 'AMD MI308X 192 GB HBM',
    'best_checkpoint': 'checkpoint-450',
    'served_model': 'soultuner-planner-v4.2-35b',
    'warm_e2e_seconds': 24.56,
    'result_source': 'model_candidate_guarded',
    'execution_weights': {'graph': 0.25, 'dense': 0.75},
    'thinking_nonempty': 0,
}
print('已验证部署快照：')
print(json.dumps(verified_deployment, ensure_ascii=False, indent=2))

endpoint = os.getenv('SOULTUNER_PLANNER_ENDPOINT', '').strip()
if not endpoint:
    print('SKIP: 未配置真实 Planner endpoint，确定性 smoke test 已完成。')
else:
    query = '低音更重、鼓点更大的歌'
    protocol = os.getenv('SOULTUNER_PLANNER_PROTOCOL', 'openai').strip().casefold()
    if protocol == 'openai':
        try:
            from deploy.modelscope_space.prompt_v42 import STUDENT_SYSTEM_PROMPT_V4_2 as system_prompt
        except Exception:
            system_prompt = (
                '你是音乐检索 Planner。只输出 JSON；task_mode 为 recommendation；'
                'lane_policy 的 graph/dense/web 只能是 required、optional、off；'
                '听感、音色、乐器或参考歌曲相似度必须令 dense=required。'
            )
        request_body = {
            'model': os.getenv('SOULTUNER_PLANNER_MODEL', 'soultuner-planner-v4.2-35b'),
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': json.dumps({'current_input': query}, ensure_ascii=False)},
            ],
            'temperature': 0, 'max_tokens': 1024, 'stream': False, 'enable_thinking': False,
        }
    else:
        request_body = {'current_input': query, 'context': {}}
    payload = json.dumps(request_body, ensure_ascii=False).encode()
    headers = {'Content-Type': 'application/json'}
    token = os.getenv('SOULTUNER_PLANNER_TOKEN', '').strip()
    if token:
        headers['Authorization'] = f'Bearer {token}'
    req = urllib.request.Request(endpoint, data=payload, headers=headers, method='POST')
    try:
        started = time.time()
        timeout = float(os.getenv('SOULTUNER_PLANNER_TIMEOUT', '180'))
        with urllib.request.urlopen(req, timeout=timeout) as response:
            raw = json.loads(response.read().decode())
        if protocol == 'openai':
            content = str(raw['choices'][0]['message']['content']).strip()
            content = re.sub(r'^<think>\s*</think>\s*', '', content, count=1, flags=re.S)
            fenced = re.fullmatch(r'```(?:json)?\s*(.*?)\s*```', content, flags=re.S | re.I)
            candidate = json.loads(fenced.group(1) if fenced else content)
        else:
            candidate = raw.get('decision', raw) if isinstance(raw, dict) else None
        final, notes = guard_candidate(query, candidate)
        print(f'endpoint latency: {time.time() - started:.2f}s')
        print(notes)
        print(json.dumps(final, ensure_ascii=False, indent=2))
    except Exception as exc:
        final = safe_plan(query)
        print(f'端点失败，已安全降级：{type(exc).__name__}')
        print(final['lane_policy'])

已验证部署快照：
{
  "run_id": "planner-v4.2-35b-2ep-20260810T164032Z",
  "gpu": "AMD MI308X 192 GB HBM",
  "best_checkpoint": "checkpoint-450",
  "served_model": "soultuner-planner-v4.2-35b",
  "warm_e2e_seconds": 24.56,
  "result_source": "model_candidate_guarded",
  "execution_weights": {
    "graph": 0.25,
    "dense": 0.75
  },
  "thinking_nonempty": 0
}
SKIP: 未配置真实 Planner endpoint，确定性 smoke test 已完成。


## 7. 自动化自检与发布清单

In [7]:
checks = {
    'mood_is_dense_primary': safe_plan('我心情很差，想听温暖治愈的歌')['execution']['profile'] == 'dense_primary',
    'acoustic_is_dense_only': safe_plan('bass 更重，鼓声更大')['execution']['profile'] == 'dense_only',
    'genre_is_graph_only': safe_plan('找一些爵士')['execution']['profile'] == 'graph_only',
    'unresolved_reference_clarifies': safe_plan('给我和刚刚那首歌相似的')['response_mode'] == 'clarify',
    'guidance_has_no_retrieval': safe_plan('怎么导入我的歌单')['execution']['profile'] == 'no_retrieval',
}
for name, passed in checks.items():
    print(('PASS' if passed else 'FAIL'), name)
assert all(checks.values())
print('\n全部 smoke checks 通过。发布时请确认：')
print('1. 携带“AMD GPU激励计划”标签；2. 不上传私有数据和 Secret；3. app.py 使用 7860；4. 添加 AMD GPU 截图。')

PASS mood_is_dense_primary
PASS acoustic_is_dense_only
PASS genre_is_graph_only
PASS unresolved_reference_clarifies
PASS guidance_has_no_retrieval

全部 smoke checks 通过。发布时请确认：
1. 携带“AMD GPU激励计划”标签；2. 不上传私有数据和 Secret；3. app.py 使用 7860；4. 添加 AMD GPU 截图。


## 结论

SoulTuner 把模型理解、Lane 角色和确定性执行分开：当前 4070 电脑可选 Qwen3.7 Plus 云端档位，AMD MI308X 或获批创空间切到训练后的 35B endpoint。模型切换不改业务代码，Policy Guard 与 Compiler 始终保持一致。Notebook 展示的是完整项目中最容易复现和审核的 Planner 部署切片；Graph、音频 Dense、融合、记忆与反馈仍由 SoulTuner-Agent 主工程承接。